In [1]:
!git clone https://github.com/keiranoluv/final_project
!git clone https://github.com/PaddlePaddle/PaddleOCR.git

Cloning into 'final_project'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 38 (delta 8), reused 32 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 18.64 KiB | 6.21 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Cloning into 'PaddleOCR'...
remote: Enumerating objects: 353017, done.
remote: Counting objects: 100% (1141/1141), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 353017 (delta 1064), reused 991 (delta 991), pack-reused 351876 (from 3)
Receiving objects: 100% (353017/353017), 1.87 GiB | 35.48 MiB/s, done.
Resolving deltas: 100% (279208/279208), done.


In [2]:
%cd /kaggle/working/PaddleOCR

!python -m pip install -q -r requirements.txt
!python -m pip install -q paddlepaddle-gpu==3.3.0 \
  -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ \
  --no-deps

/kaggle/working/PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 69.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 867.6 kB/s eta 0:00:00


In [3]:
import csv
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2")
OUTPUT_ROOT = Path("/kaggle/working/mthv2_labels")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    src = DATASET_ROOT / f"{split}.tsv"
    dst = OUTPUT_ROOT / f"{split}.txt"

    count = 0

    with src.open("r", encoding="utf-8") as fin, \
         dst.open("w", encoding="utf-8") as fout:

        reader = csv.DictReader(fin, delimiter="\t")

        for row in reader:
            fout.write(f'{row["image_path"]}\t{row["text"]}\n')
            count += 1

    print(f"{split}: {count:,} samples -> {dst}")

train: 72,563 samples -> /kaggle/working/mthv2_labels/train.txt
val: 7,753 samples -> /kaggle/working/mthv2_labels/val.txt
test: 25,262 samples -> /kaggle/working/mthv2_labels/test.txt


## B3-Control — Continued Fine-Tuning Without Oversampling

### 1. Objective

Mục tiêu của experiment này là kiểm tra liệu phần cải thiện quan sát được ở B3 có đến từ rare-character oversampling hay đơn giản do mô hình B2 được fine-tune thêm.

---

### 2. Training setup

Mô hình được khởi tạo từ **B2 best checkpoint** và tiếp tục fine-tune trên **training set gốc**, không sử dụng oversampling.

```text
Initialization      : B2 best checkpoint
Training set        : original train.txt
Oversampling        : No
Epochs              : 10
Learning rate       : 0.00005
Warmup epoch        : 0
Vocabulary          : B2 expanded vocabulary
Validation          : original val.txt
Test                : untouched test.tsv

In [5]:
%cd /kaggle/working/PaddleOCR

!python -m paddle.distributed.launch \
  --gpus "0,1" \
  tools/train.py \
  -c configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml \
  -o \
  Global.pretrained_model=/kaggle/input/models/zephyrvn/b2-best-checkpoint/other/default/1/best_model_b2_open_vocab.pdparams \
  Global.character_dict_path=/kaggle/input/datasets/zephyrvn/vocabylary-expanded/ppocrv5_mthv2_expanded.txt \
  Global.epoch_num=10 \
  Global.save_model_dir=/kaggle/working/final_project/outputs/B3_rare_char_ft_10ep \
  Global.eval_batch_step="[0,100]" \
  Optimizer.lr.learning_rate=0.00005 \
  Optimizer.lr.warmup_epoch=0 \
  Train.loader.batch_size_per_card=64 \
  Train.sampler.first_bs=64 \
  Train.dataset.data_dir=/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2 \
  Train.dataset.label_file_list='["/kaggle/working/mthv2_labels/train.txt"]' \
  Eval.dataset.data_dir=/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2 \
  Eval.dataset.label_file_list='["/kaggle/working/mthv2_labels/val.txt"]'

/kaggle/working/PaddleOCR
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
LAUNCH INFO 2026-08-10 18:09:04,183 -----------  Configuration  ----------------------
LAUNCH INFO 2026-08-10 18:09:04,183 auto_cluster_config: 0
LAUNCH INFO 2026-08-10 18:09:04,184 auto_parallel_config: None
LAUNCH INFO 2026-08-10 18:09:04,184 auto_tuner_json: None
LAUNCH INFO 2026-08-10 18:09:04,184 devices: 0,1
LAUNCH INFO 2026-08-10 18:09:04,184 elastic_level: -1
LAUNCH INFO 2026-08-10 18:09:04,184 elastic_timeout: 30
LAUNCH INFO 2026-08-10 18:09:04,184 enable_gpu_log: True
LAUNCH INFO 2026-08-10 18:09:04,184 gloo_port: 6767
LAUNCH INFO 2026-08-10 18:09:04,184 host: None
LAUNCH INFO 2026-08-10 18:09:04,184 ips: None
LAUNCH INFO 2026-08-10 